<a href="https://colab.research.google.com/github/UlaStats/MSc-project-pipe-burst-prediction/blob/main/MSc_project_data_manipulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import packages required

from google.colab import drive
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats

In [ ]:
# mount drive

drive.mount('/content/drive')

Mounted at /content/drive


# Data manipulation

In [ ]:
# import data

assets_cleaned = pd.read_csv("/content/drive/MyDrive/MSc project/assets_cleaned.csv", encoding ='latin1')

bursts_cleaned = pd.read_csv("/content/drive/MyDrive/MSc project/bursts_cleaned.csv", encoding ='latin1')

soil_cleaned = pd.read_csv("/content/drive/MyDrive/MSc project/soil_cleaned.csv", encoding ='latin1')

### Joining

In [ ]:
# merging assets, bursts and soil together

assets_bursts = pd.merge(assets_cleaned, bursts_cleaned, how = "left", on = "Asset.ID")

assets_bursts = assets_bursts[pd.notnull(assets_bursts["Raised.Date"])] # removing assets without bursts

assets_bursts_soil = pd.merge(assets_bursts, soil_cleaned, how = "left", on = "Asset.ID")

assets_bursts_soil = assets_bursts_soil[pd.notnull(assets_bursts_soil['Soil'])] # removing assets without soil

assets_bursts_soil["Raised.Date"] = pd.to_datetime(assets_bursts_soil["Raised.Date"], dayfirst=True) # transofrms burst date to date format

/tmp/ipykernel_1448/3104486003.py:11: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  assets_bursts_soil["Raised.Date"] = pd.to_datetime(assets_bursts_soil["Raised.Date"], dayfirst=True) # transofrms burst date to date format


### Transformations

In [ ]:
# identify assets with 2 or more bursts

counts_2 = assets_bursts_soil["Asset.ID"].value_counts()
frequent_bursters_2 = counts_2[counts_2 >= 2].index


# identify assets with 3 or more bursts

counts_3 = assets_bursts_soil["Asset.ID"].value_counts()
frequent_bursters_3 = counts_3[counts_3 >= 3].index


In [ ]:
# filter data for assets with 2 or more bursts only

assets_bursts_soil_frequent_2 = assets_bursts_soil[assets_bursts_soil["Asset.ID"].isin(frequent_bursters_2)]

# filter data for assets with 3 or more bursts

assets_bursts_soil_frequent_3 = assets_bursts_soil[assets_bursts_soil['Asset.ID'].isin(frequent_bursters_3)]


In [ ]:
# identify the latest 2 bursts for each asset

index_2 = assets_bursts_soil_frequent_2.groupby("Asset.ID")["Raised.Date"].nlargest(2).reset_index(level = 0, drop = True).index

# identify the latest 3 bursts for each asset

index_3 = assets_bursts_soil_frequent_3.groupby("Asset.ID")["Raised.Date"].nlargest(3).reset_index(level = 0, drop = True).index

In [ ]:
# subset data to contain only latest 2 bursts

assets_bursts_soil_subset_2 = assets_bursts_soil_frequent_2.loc[index_2]

# subset data to contain only latest 3 bursts

assets_bursts_soil_subset_3 = assets_bursts_soil_frequent_3.loc[index_3]

In [ ]:
# sort using Asset ID and Date

assets_bursts_soil_subset_2 = assets_bursts_soil_subset_2.sort_values(by = ["Asset.ID", "Raised.Date"])

assets_bursts_soil_subset_3 = assets_bursts_soil_subset_3.sort_values(by = ["Asset.ID", "Raised.Date"])


In [ ]:
# introduce a pivot column

burst_2 = ["First", "Second"] * (int(assets_bursts_soil_subset_2.shape[0]/2))

assets_bursts_soil_subset_2["Burst"] = burst_2

burst_3 = ["First", "Second", "Third"] * (int(assets_bursts_soil_subset_3.shape[0]/3))

assets_bursts_soil_subset_3["Burst"] = burst_3

In [ ]:
# pivot date column

pivoted_column_2 = assets_bursts_soil_subset_2.pivot(values = "Raised.Date", columns = "Burst", index = "Asset.ID").reset_index()

pivoted_column_3 = assets_bursts_soil_subset_3.pivot(values = "Raised.Date", columns = "Burst", index = "Asset.ID").reset_index()



In [ ]:
# merge pivoted column with the original assets and soil data

attributes_2 = pd.merge(pivoted_column_2, assets_cleaned, how = "inner", on = "Asset.ID")

attributes_2 = pd.merge(attributes_2, soil_cleaned, on = "Asset.ID", how = "inner")


attributes_3 = pd.merge(pivoted_column_3, assets_cleaned, how = "inner", on = "Asset.ID")

attributes_3 = pd.merge(attributes_3, soil_cleaned, on = "Asset.ID", how = "inner")

In [ ]:
# calculate and add time-to-break attribute

time_to_next_break_2 = attributes_2["Second"] - attributes_2["First"]

attributes_2["Time-to-break"] = time_to_next_break_2

attributes_2["Time-to-break"] = attributes_2['Time-to-break'].astype("int64")/ (1e9 * 60 * 60 * 24) # convert to numeric data type (converted from nano seconds to days)


time_to_next_break_3 = attributes_3['Third'] - attributes_3['Second']

attributes_3["Time-to-break"] = time_to_next_break_3

attributes_3["Time-to-break"] = attributes_3['Time-to-break'].astype("int64")/ (1e9 * 60 * 60 * 24)



In [ ]:
# calculate time between previous break

time_between_breaks = attributes_3["Second"] - attributes_3["First"]

attributes_3["Time-between-breaks"] = time_between_breaks

attributes_3["Time-between-breaks"] = attributes_3['Time-between-breaks'].astype("int64")/ (1e9 * 60 * 60 * 24)

In [ ]:
# calculate the no. of previous failures

previous_bursts_2 = assets_bursts_soil_frequent_2['Asset.ID'].value_counts() - 1 # one substracted as that corresponds to future burst

previous_bursts_3 = assets_bursts_soil_frequent_3['Asset.ID'].value_counts() - 1 # one substracted as that corresponds to future burst

In [ ]:
# merge with the no. of previous failures

attributes_2 = pd.merge(attributes_2, previous_bursts_2, how = "inner", on = "Asset.ID")
attributes_2 = attributes_2.rename(columns={"count":"previous fails"}) # change column name

attributes_3 = pd.merge(attributes_3, previous_bursts_3, how = "inner", on = "Asset.ID")
attributes_3 = attributes_3.rename(columns={"count":"previous fails"}) # change column name

In [ ]:
# save attributes

attributes_2.to_csv("/content/drive/MyDrive/MSc project/attributes_2.csv", encoding ='latin1', index=False)

attributes_3.to_csv("/content/drive/MyDrive/MSc project/attributes_3.csv", encoding ='latin1', index=False)